# Team Model (xG)

The fitted team model, which is `xg` by default: attack and defence ratings from
the expected goals in past matches, wrapped in a Conway-Maxwell-Poisson that
turns a predicted mean into the distribution over goal counts the points
calculation needs.

Those ratings are point estimates, and multiplicative around a league average of
one - an attack of 1.2 creates a fifth more than the league does, against the
same defence at the same venue - so there are no posterior intervals to draw
here. For the goals-fitted model this replaced as the default - bpl's
Dixon-Coles, which does have a posterior - see `team_model_goals.ipynb`.

`tools/team_ratings.py` prints the same ratings from the command line.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from airsenal.db.queries.fixtures import get_fixtures_for_gameweeks
from airsenal.db.queries.gameweeks import get_max_gameweek, next_gameweek
from airsenal.db.queries.teams import get_teams_for_season
from airsenal.db.session import get_session
from airsenal.game.scoring import MAX_GOALS
from airsenal.game.season import CURRENT_SEASON
from airsenal.prediction.team_models.fitting import get_fitted_team_model
from airsenal.prediction.team_models.scorelines import (
    DEFAULT_GOAL_DISPERSION,
    POISSON_DISPERSION,
    conway_maxwell_pmf,
)

## Fit Model

In [ ]:
session = get_session()
CURRENT_TEAMS = get_teams_for_season(CURRENT_SEASON, session)
NEXT_GAMEWEEK = next_gameweek()
model_team = get_fitted_team_model(NEXT_GAMEWEEK, CURRENT_SEASON, session)

# The ratings live on the model that the TEAM_MODELS entry wrapped; the wrapper
# itself only knows how to turn a mean into goal counts.
xg_model = model_team.model
print(
    f"Fitted on every result before {CURRENT_SEASON} GW{NEXT_GAMEWEEK}: "
    f"league {xg_model.home_mean:.2f} xG at home, {xg_model.away_mean:.2f} away"
)

In [ ]:
def ratings(fitted, teams=None):
    """The fitted attack and defence ratings as a frame, best net rating first.

    The fitting window reaches back through earlier seasons, so teams that have
    since been relegated are rated too - they are what the current teams' records
    were built against. `teams` picks out the ones playing this season.
    """
    model = fitted.model
    frame = pd.DataFrame(
        {"attack": pd.Series(model.attack), "defence": pd.Series(model.defence)}
    )
    if teams is not None:
        frame = frame.loc[list(teams)]
    frame["xg_for"] = model.home_mean * frame["attack"]
    frame["xg_against"] = model.away_mean * frame["defence"]
    frame["net"] = frame["attack"] / frame["defence"]
    return frame.sort_values("net", ascending=False)


def fit_at(gameweek, season=CURRENT_SEASON):
    """The model as it would have been before `gameweek`, for comparing dates."""
    return get_fitted_team_model(gameweek, season, session)


def label_points(axis, x, y, labels, fontsize=11):
    """Annotate points that sit on top of each other.

    Twenty teams land in a tight cluster, and labelling every one of them the
    same way up and to the right makes the middle of it unreadable, so the
    offsets cycle instead. Some of them still touch; nothing here solves for a
    layout.
    """
    # Cycled by where a point sits left to right rather than by the order the
    # labels arrive in, so that two teams side by side never get the same one.
    offsets = [(8, 5), (8, -12), (-30, 5), (-30, -12)]
    place = {point: order for order, point in enumerate(np.argsort(np.asarray(x)))}
    for i, (x_i, y_i, label) in enumerate(zip(x, y, labels, strict=True)):
        axis.annotate(
            label,
            (x_i, y_i),
            textcoords="offset points",
            xytext=offsets[place[i] % len(offsets)],
            fontsize=fontsize,
        )


# Every goal count the wrapper has anything to say about.
goals = np.arange(MAX_GOALS + 1)

current = ratings(model_team, CURRENT_TEAMS)
# One colour per team, so that the views below can be read against each other.
TEAM_COLOUR = dict(zip(current.index, plt.get_cmap("tab20").colors, strict=False))
current.round(3)

## Match Outcome Predictions

`predict_score_n_proba` asks the wrapper for one side's goal counts;
`predict_outcome_proba` takes the two sides as sequences and wants a list even
for a single fixture.

In [ ]:
fixtures = get_fixtures_for_gameweeks([NEXT_GAMEWEEK], CURRENT_SEASON, session)

nrow = int(np.ceil(len(fixtures) / 2))
fig, ax = plt.subplots(nrow, 2, figsize=(9, 2.9 * nrow), sharey=True)
ax = ax.flatten()

for i, f in enumerate(fixtures):
    prob_home = model_team.predict_score_n_proba(goals, f.home_team, f.away_team)
    prob_away = model_team.predict_score_n_proba(
        goals, f.away_team, f.home_team, home=False
    )
    outcome = model_team.predict_outcome_proba([f.home_team], [f.away_team])
    # The model treats the two goal counts as independent, which is what
    # predict_outcome_proba assumes as well, so the likeliest scoreline is the
    # largest entry of the outer product.
    joint = np.outer(prob_home, prob_away)
    likeliest = np.unravel_index(joint.argmax(), joint.shape)

    ax[i].bar(goals, prob_home, facecolor="none", edgecolor="b", label=f.home_team)
    ax[i].bar(goals, prob_away, facecolor="none", edgecolor="r", label=f.away_team)
    ax[i].set_xlim([-1, MAX_GOALS])
    bs, be = r"$\bf{", r"}$"
    ax[i].set_title(
        f"{bs}{f.home_team}{be} {bs}vs.{be} {bs}{f.away_team}{be}\n"
        f"{goals @ prob_home:.2f} goals vs. {goals @ prob_away:.2f} goals\n"
        f"Home win: {outcome['home_win'][0]:.2f}, "
        f"Draw: {outcome['draw'][0]:.2f}, "
        f"Away win: {outcome['away_win'][0]:.2f}\n"
        f"Likeliest score: {likeliest[0]}-{likeliest[1]} ({joint[likeliest]:.2f})",
        fontsize=10,
    )
    ax[i].set_ylabel("Probability")
    ax[i].set_xlabel("Goals")
    ax[i].legend()

for spare in ax[len(fixtures) :]:
    spare.set_visible(False)

fig.suptitle(
    f"Predicted Match Outcomes for GW{NEXT_GAMEWEEK}, Season {CURRENT_SEASON}",
    weight="bold",
)
fig.tight_layout()

## Team Strengths / Model Parameters

Attack against defence, both relative to a league average of one. Better teams
are to the bottom right: creating more than average and conceding less.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.set_aspect("equal")
ax.scatter(current["attack"], current["defence"], s=60)
ax.axhline(1, color="k", linewidth=0.75)
ax.axvline(1, color="k", linewidth=0.75)

label_points(ax, current["attack"], current["defence"], current.index)

ax.set_xlabel("attack", fontsize=14)
ax.set_ylabel("defence", fontsize=14)
ax.set_title(f"{CURRENT_SEASON} GW{NEXT_GAMEWEEK}: league average is one on both axes")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 6))

for axis, column, title in (
    (ax[0], "attack", "Attack (expected goals created, league = 1)"),
    (ax[1], "defence", "Defence (expected goals conceded, league = 1)"),
):
    # Both are ordered so that the top of the chart is the better team, which
    # for defence means the smaller number.
    ordered = current.sort_values(column, ascending=column == "defence")
    axis.barh(
        ordered.index,
        ordered[column] - 1,
        left=1,
        color="tab:blue" if column == "attack" else "tab:red",
    )
    axis.axvline(1, color="k", linewidth=0.75)
    axis.invert_yaxis()
    axis.set_title(title)
    axis.set_xlabel(column)

fig.tight_layout()

### Upcoming fixtures

The ratings applied to the fixtures actually coming up: expected goals created
minus expected goals conceded, per team per gameweek, green being the easier
fixture. Home fixtures are in capitals. These are the ratings as they stand now,
so the further out a column is the more it says about the opponent and the less
about anyone's form by then - a double gameweek adds its two fixtures together
and a blank one is left empty.

In [ ]:
HORIZON = 6
gameweeks = [
    gameweek
    for gameweek in range(NEXT_GAMEWEEK, NEXT_GAMEWEEK + HORIZON)
    if gameweek <= get_max_gameweek(season=CURRENT_SEASON, dbsession=session)
]

net = {}
opponents = {}
for gameweek in gameweeks:
    for f in get_fixtures_for_gameweeks([gameweek], CURRENT_SEASON, session):
        for team, opponent, at_home in (
            (f.home_team, f.away_team, True),
            (f.away_team, f.home_team, False),
        ):
            key = (team, gameweek)
            scored = model_team.predict_expected_goals(team, opponent, home=at_home)
            conceded = model_team.predict_expected_goals(
                opponent, team, home=not at_home
            )
            net[key] = net.get(key, 0.0) + scored - conceded
            venue = opponent.upper() if at_home else opponent.lower()
            opponents[key] = f"{opponents[key]} {venue}" if key in opponents else venue

difficulty = pd.DataFrame(
    [
        [net.get((team, gameweek), np.nan) for gameweek in gameweeks]
        for team in current.index
    ],
    index=current.index,
    columns=gameweeks,
)
annotations = pd.DataFrame(
    [
        [opponents.get((team, gameweek), "") for gameweek in gameweeks]
        for team in current.index
    ],
    index=current.index,
    columns=gameweeks,
)
sum_net = difficulty.sum(axis=1).sort_values(ascending=False).index
difficulty = difficulty.loc[sum_net, :]
annotations = annotations.loc[sum_net, :]
fig, ax = plt.subplots(1, 1, figsize=(2 + 1.4 * len(gameweeks), 8))
sns.heatmap(
    difficulty,
    annot=annotations,
    fmt="",
    annot_kws={"fontsize": 8},
    cmap="RdYlGn",
    center=0,
    cbar_kws={"label": "expected goals created minus conceded"},
    ax=ax,
)
ax.set_xlabel("Gameweek")
ax.set_ylabel("")
ax.set_title(f"Fixtures from GW{gameweeks[0]} to GW{gameweeks[-1]}, {CURRENT_SEASON}")
fig.tight_layout()

### From a mean to a distribution over goal counts

The xG model predicts a mean and nothing else, so the wrapper supplies the
spread. `DEFAULT_GOAL_DISPERSION` is the held-out optimum and is above one,
which makes the distribution narrower than a Poisson at the same mean - and a
clean sheet correspondingly less likely.

In [ ]:
mean = xg_model.home_mean
shipped = conway_maxwell_pmf(mean, DEFAULT_GOAL_DISPERSION)[0]
poisson = conway_maxwell_pmf(mean, POISSON_DISPERSION)[0]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(
    goals - 0.2,
    poisson,
    width=0.4,
    color="grey",
    label=f"Poisson (dispersion {POISSON_DISPERSION:g})",
)
ax[0].bar(
    goals + 0.2,
    shipped,
    width=0.4,
    color="tab:blue",
    label=f"shipped (dispersion {DEFAULT_GOAL_DISPERSION:g})",
)
ax[0].set_xlim(-0.7, 6.7)
ax[0].set_xticks(range(7))
ax[0].set_xlabel("Goals")
ax[0].set_ylabel("Probability")
ax[0].set_title(f"Goals at the league home mean of {mean:.2f}")
ax[0].legend()

means = np.linspace(0.4, 3.0, 60)
clean_sheet_poisson = conway_maxwell_pmf(means, POISSON_DISPERSION)[:, 0]
clean_sheet_shipped = conway_maxwell_pmf(means, DEFAULT_GOAL_DISPERSION)[:, 0]
ax[1].plot(means, clean_sheet_poisson, color="grey", label="Poisson")
ax[1].plot(means, clean_sheet_shipped, color="tab:blue", label="shipped")
ax[1].set_xlabel("Opponent's expected goals")
ax[1].set_ylabel("P(clean sheet)")
ax[1].set_title("A clean sheet is less likely than a Poisson would say")
ax[1].legend()

fig.tight_layout()
print(
    "Clean sheet against an average away side: "
    f"{clean_sheet_shipped[np.abs(means - xg_model.away_mean).argmin()]:.3f} "
    "against a Poisson's "
    f"{clean_sheet_poisson[np.abs(means - xg_model.away_mean).argmin()]:.3f}"
)

### Change since start of season

In [ ]:
start = ratings(fit_at(1), CURRENT_TEAMS).reindex(current.index)
change = pd.DataFrame(
    {
        "attack": current["attack"] - start["attack"],
        "defence": current["defence"] - start["defence"],
    }
)
# How far a team has moved in the attack-defence plane, in either direction:
# how much of a surprise its results have been.
change["surprise"] = np.sqrt(change["attack"] ** 2 + change["defence"] ** 2)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.set_aspect("equal")

for team in current.index:
    ax.arrow(
        start.loc[team, "attack"],
        start.loc[team, "defence"],
        change.loc[team, "attack"],
        change.loc[team, "defence"],
        width=0.002,
        length_includes_head=True,
        color="tab:blue",
    )

label_points(ax, current["attack"], current["defence"], current.index)
ax.axhline(1, color="k", linewidth=0.75)
ax.axvline(1, color="k", linewidth=0.75)
ax.set_xlabel("attack", fontsize=14)
ax.set_ylabel("defence", fontsize=14)
ax.set_title(f"GW{NEXT_GAMEWEEK} vs. GW1")
plt.show()

print("Ratings change since the start of the season:")
display(change.sort_values("surprise", ascending=False).round(3))

### Change since last gameweek

A team with three matches played is mostly the league average and one with
thirty is mostly itself, so the promoted sides move furthest on one result -
`prior_matches` is a number of matches at any time weighting.

In [ ]:
previous = ratings(fit_at(NEXT_GAMEWEEK - 1), CURRENT_TEAMS).reindex(current.index)
last_gameweek = pd.DataFrame(
    {
        "attack": current["attack"] - previous["attack"],
        "defence": current["defence"] - previous["defence"],
    }
)
last_gameweek["surprise"] = np.sqrt(
    last_gameweek["attack"] ** 2 + last_gameweek["defence"] ** 2
)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.set_aspect("equal")
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)

for team, row in last_gameweek.iterrows():
    ax.text(
        row["attack"],
        row["defence"],
        team,
        horizontalalignment="center",
        verticalalignment="center",
        fontsize=10,
    )
ax.set_xlim(
    1.2 * last_gameweek["attack"].min() - 0.005,
    1.2 * last_gameweek["attack"].max() + 0.005,
)
ax.set_ylim(
    1.2 * last_gameweek["defence"].min() - 0.005,
    1.2 * last_gameweek["defence"].max() + 0.005,
)

ax.set_xlabel(r"$\Delta$(attack)", fontsize=11)
ax.set_ylabel(r"$\Delta$(defence)", fontsize=11)
ax.set_title(f"GW{NEXT_GAMEWEEK} vs. GW{NEXT_GAMEWEEK - 1}")
plt.show()

print(f"Ratings updates GW{NEXT_GAMEWEEK} vs. GW{NEXT_GAMEWEEK - 1}:")
display(last_gameweek.sort_values("surprise", ascending=False).round(3))

### Changes during season (slow)

One fit per gameweek so far. Each is fitted only on what had been played by
then, so this is what the model believed at the time rather than a smoothed
version of what it believes now.

In [ ]:
history = {
    gameweek: ratings(fit_at(gameweek), CURRENT_TEAMS)
    for gameweek in range(1, NEXT_GAMEWEEK + 1)
}
attack = pd.DataFrame(
    {gameweek: frame["attack"] for gameweek, frame in history.items()}
).reindex(current.index)
defence = pd.DataFrame(
    {gameweek: frame["defence"] for gameweek, frame in history.items()}
).reindex(current.index)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 9))
for team in current.index:
    ax.plot(
        attack.loc[team],
        defence.loc[team],
        marker="none",
        color=TEAM_COLOUR[team],
        label=team,
    )
label_points(
    ax,
    attack.iloc[:, -1],
    defence.iloc[:, -1],
    attack.index,
)
ax.set_xlabel("Attack")
ax.set_ylabel("Defence")
ax.axhline(1, color="k", linewidth=0.5)
ax.axvline(1, color="k", linewidth=0.5)
ax.axis("equal")
ax.set_title(f"Ratings through {CURRENT_SEASON}, GW1 to GW{NEXT_GAMEWEEK}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharex=True)
for team in current.index:
    ax[0].plot(
        attack.columns,
        attack.loc[team],
        marker="o",
        color=TEAM_COLOUR[team],
        label=team,
    )
    ax[1].plot(defence.columns, defence.loc[team], marker="o", color=TEAM_COLOUR[team])

for axis, title in ((ax[0], "Attack"), (ax[1], "Defence")):
    axis.axhline(1, color="k", linewidth=0.5)
    axis.set_xticks(list(attack.columns))
    axis.set_xlabel("Gameweek")
    axis.set_title(title)

fig.legend(
    loc="outside lower center",
    ncol=int(np.ceil(len(current) / 2)),
    bbox_to_anchor=(0.5, -0.15),
)